# Day 35 — Model evaluation & cross-validation
Objectives:
- Metrics (classification/regression).
- k-fold cross-validation.
- Bias/variance intuition.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
X,y = load_breast_cancer(return_X_y=True)
pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
scores = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
scores.mean(), scores


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — metrics, resampling design, and honest generalization estimates

### Mental model

Evaluation begins with the decision, not a convenient default metric.
A confusion matrix records counts at one decision threshold; precision
asks whether positive predictions are reliable, while recall asks how
many actual positives were found. Ranking metrics such as ROC AUC and
average precision evaluate scores across thresholds but do not select
an operating threshold for you.

Cross-validation repeatedly changes which observations train and
validate a model. Its validity depends on the split unit matching the
independence structure. Repeated records from one person, device, or
time period must not leak across folds merely because ordinary K-fold
code runs.

### Read the API before running it

- **`confusion_matrix(y_true, y_pred)`:** returns outcome counts at the selected threshold; inspect label order before unpacking.
- **`cross_validate(estimator, X, y, cv=..., scoring=...)`:** fits a fresh estimator per fold and can report multiple metrics and timing.
- **splitter objects:** encode random, stratified, grouped, or temporal assumptions; `cv=5` is not a universal design.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — derive precision and recall from outcome counts

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Class `1` is the positive class and each error type has a known operational consequence.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score

y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0])
y_pred = np.array([1, 0, 1, 0, 1, 0, 0, 0])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
print({"tn": tn, "fp": fp, "fn": fn, "tp": tp,
       "precision": precision, "recall": recall})
assert precision == tp / (tp + fp)
assert recall == tp / (tp + fn)

**Expected observation:** Precision is 2/3 and recall is 1/2; the two metrics answer different questions.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — keep entity groups out of each other's folds

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The group ID captures the dependence that would otherwise create optimistic validation.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold

groups = np.repeat(["A", "B", "C", "D"], 3)
X = np.arange(groups.size).reshape(-1, 1)
y = np.tile([0, 1, 0], 4)
splitter = GroupKFold(n_splits=4)

for train_idx, valid_idx in splitter.split(X, y, groups):
    train_groups = set(groups[train_idx])
    valid_groups = set(groups[valid_idx])
    assert train_groups.isdisjoint(valid_groups)
print("all validation groups were unseen during training")

**Expected observation:** Each entity appears entirely in training or validation for a fold, never both.

### Debugging and practice ramp

**Common mistake:** Choosing the metric and split after seeing which combination makes the model look best.

**Diagnostic:** Write the row grain, positive class, prediction horizon, grouping key, and error costs before constructing the splitter or scorer.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define metrics, resampling design, and honest generalization estimates in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not compare scores produced from different rows, folds, positive labels, thresholds, or metric definitions.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Evaluate the pipeline with `accuracy`, `f1`, and `roc_auc`.

**Verify:** Practice 1 — metrics, resampling design, and honest generalization estimates — on the same declared stratified folds, print every accuracy, F1, and ROC-AUC fold score plus mean and standard deviation; include positive-class support and use probability/decision scores—not hard labels—for ROC AUC.

2. Compare the variability from 5-fold and 10-fold cross-validation.

**Verify:** Practice 2 — metrics, resampling design, and honest generalization estimates — use identical data, estimator, scorer, shuffle policy, and seed for 5 and 10 folds; print both score vectors, means, and standard deviations, and confirm every row appears in validation exactly once per run.

3. Explain leakage risks and how placing preprocessing in a pipeline helps.

**Verify:** Practice 3 — metrics, resampling design, and honest generalization estimates — show fold indices or a fit counter proving preprocessing is fitted separately inside each training fold; contrast with one deliberately pre-fitted transformation and explain why its validation score is contaminated.

### Progressive hints

1. Keep folds identical across metrics. Ask whether each metric uses predicted
   labels, scores, or probabilities.
2. Create two `StratifiedKFold` objects with shuffle and fixed seeds. Compare
   both mean and spread; do not infer a universal rule from one dataset.
3. Sketch what the scaler would know if it were fit before the folds existed.
   Repeat the reasoning for imputation and feature selection.

### Additional mastery practice

Match metrics and resampling to the decision being modeled. Keep thresholds, groups, time, and hyperparameter selection inside honest validation boundaries.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Threshold analysis:** Using one fixed validation score vector, compare confusion matrices at thresholds 0.2, 0.5, and 0.8. Explain which errors increase as the threshold rises and why ROC AUC stays unchanged.
   **Progressive hint:** A higher positive threshold generally reduces predicted positives: false positives fall while false negatives rise. Ranking scores do not change.

**Verify:** Threshold analysis — for one fixed validation score vector, print TP/FP/TN/FN at 0.2, 0.5, and 0.8; assert predicted-positive count cannot rise with threshold and ROC AUC is identical because scores did not change.

5. **Grouped resampling:** Design cross-validation for repeated measurements from the same patient or customer. Demonstrate how ordinary StratifiedKFold can place one entity in both training and validation.
   **Progressive hint:** Use `StratifiedGroupKFold` when both label balance and entity separation matter; assert that train and validation group sets are disjoint.

**Verify:** Grouped resampling — print train/validation entity IDs for every grouped fold and assert their intersections are empty; also exhibit at least one ordinary StratifiedKFold split where the same entity appears on both sides.

6. **Selection-bias debugging:** Explain why reporting `GridSearchCV.best_score_` as final performance is optimistic. Sketch a nested cross-validation design and distinguish it from out-of-fold predictions for one fixed model.
   **Progressive hint:** The same inner folds both select and report the best candidate. Nested CV puts the complete search inside an outer held-out fold.

**Verify:** Selection-bias debugging — print nested outer-fold scores plus mean/std and the inner best parameters per outer fold; contrast with best_score_ and state that one fixed model's out-of-fold predictions do not evaluate the selection procedure.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Threshold analysis


# Practice 5 — Grouped resampling


# Practice 6 — Selection-bias debugging
